# Driver Drowsiness Detection (Eye + Yawn) - Google Colab Notebook

This notebook runs the complete workflow end-to-end:
1. Setup dependencies
2. Load `archive.zip` dataset
3. Prepare Eye + Yawn train/valid/test splits
4. Train eye model
5. Evaluate eye model
6. Train yawn model
7. Evaluate yawn model
8. Save artifacts

> **Important:** Live webcam alert (`detect.py`) is best run on your local laptop/desktop, not in Colab.

## 0) Runtime recommendation
- Colab menu -> **Runtime -> Change runtime type -> GPU** (recommended).
- If you use CPU, training still works but will be slower.

In [ ]:
!python --version
!nvidia-smi || true

## 1) Clone repository and move to project folder

In [ ]:
# Auto-locate existing repo in Colab; clone only if needed
import os
from pathlib import Path

COMMON_REPO_DIRS = [
    Path('/content/Driver-Drowsiness-Detection-using-Deep-learning'),
    Path('/content/repo'),
]

REPO_DIR = next((p for p in COMMON_REPO_DIRS if p.exists()), None)

if REPO_DIR is None:
    REPO_URL = input('Paste your GitHub repo URL (https://github.com/<user>/<repo>.git): ').strip()
    if not REPO_URL.startswith('https://github.com/') or not REPO_URL.endswith('.git'):
        raise ValueError('Please provide a valid GitHub .git URL.')

    repo_name = REPO_URL.rsplit('/', 1)[-1].replace('.git', '')
    REPO_DIR = Path('/content') / repo_name
    clone_cmd = f'git clone {REPO_URL} {REPO_DIR}'
    print('Running:', clone_cmd)
    rc = os.system(clone_cmd)
    if rc != 0:
        raise RuntimeError('Git clone failed. Check URL access/permissions and retry.')
else:
    print('Repository already present:', REPO_DIR)

# Robustly locate project folder that contains detect.py + train.py
candidate_paths = [
    REPO_DIR / 'Driver_Drowsiness_Detection' / 'Driver_Drowsiness_Detection',
    REPO_DIR / 'Driver_Drowsiness_Detection',
    REPO_DIR,
]
PROJECT_DIR = None
for cand in candidate_paths:
    if (cand / 'detect.py').exists() and (cand / 'train.py').exists():
        PROJECT_DIR = cand
        break

if PROJECT_DIR is None:
    matches = [p.parent for p in REPO_DIR.rglob('detect.py') if (p.parent / 'train.py').exists()]
    if matches:
        PROJECT_DIR = matches[0]

if PROJECT_DIR is None:
    raise FileNotFoundError('Could not auto-detect project folder containing detect.py and train.py')

print('PROJECT_DIR =', PROJECT_DIR)
%cd $PROJECT_DIR


## 2) Install dependencies

In [ ]:
!pip -q install tensorflow==2.15.0 opencv-python-headless==4.10.0.84 pygame==2.6.1 scikit-learn

## 3) Upload `archive.zip`
Upload the dataset zip from your computer if it is not already present.

In [ ]:
from pathlib import Path
from google.colab import files

archive_path = Path('archive.zip')
if not archive_path.exists():
    print('Please upload archive.zip now...')
    uploaded = files.upload()
    if 'archive.zip' not in uploaded:
        raise FileNotFoundError('Upload must include a file named archive.zip')

assert archive_path.exists(), 'archive.zip not found after upload'
print('archive.zip ready at', archive_path.resolve())

## 4) Prepare combined eye+yawn dataset from zip

In [ ]:
!python prepare_combined_dataset.py

In [ ]:
from pathlib import Path

def count_images(folder):
    exts = {'.png','.jpg','.jpeg','.bmp','.webp'}
    p = Path(folder)
    return sum(1 for f in p.rglob('*') if f.is_file() and f.suffix.lower() in exts)

for split in ['train','valid','test']:
    for cls in ['open','closed']:
        print(f'data/{split}/{cls}:', count_images(Path('data')/split/cls))

for split in ['train','valid','test']:
    for cls in ['yawn','no_yawn']:
        print(f'data_yawn/{split}/{cls}:', count_images(Path('data_yawn')/split/cls))

## 5) Train eye model

In [ ]:
!python train.py

## 6) Evaluate eye model

In [ ]:
!python evaluate_eye.py

## 7) Train yawn model

In [ ]:
!python train_yawn.py

## 8) Evaluate yawn model

In [ ]:
!python evaluate_yawn.py

## 9) Verify output artifacts
After successful run you should see:
- `models/cnnCat2.h5` (eye model)
- `models/yawn_cnn.h5` (yawn model)

In [ ]:
from pathlib import Path
for model_file in ['models/cnnCat2.h5','models/yawn_cnn.h5']:
    p = Path(model_file)
    print(model_file, '->', 'FOUND' if p.exists() else 'MISSING')

## 10) (Optional) Save artifacts to Google Drive
Uncomment and run if you want models/checkpoints saved permanently.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/drowsiness_models
# !cp -v models/cnnCat2.h5 models/yawn_cnn.h5 /content/drive/MyDrive/drowsiness_models/

## 11) Run live detector
Run this on your local machine (recommended), because Colab webcam and `pygame` alarm are not reliable for real-time app usage.

```bash
python detect.py
```
Press **q** to stop.